In [ ]:
def extract_residuals(dataset, model_obj, region="water", model_name="Baseline", save_path=None):
    all_residuals = []
    all_sample_ids = []

    for idx, sample in enumerate(dataset):
        hr_mask = sample["hr_mask"].squeeze(0)
        water_mask = sample["water_mask"].squeeze(0)

        # -----------------------------
        # Region mask
        # -----------------------------
        if region == "water":
            region_mask = (hr_mask > 0.5) & (water_mask > 0.5)
        elif region == "non_water":
            region_mask = (hr_mask > 0.5) & (water_mask <= 0.5)
        else:
            raise ValueError("region must be 'water' or 'non_water'")

        if region_mask.sum() < 10:
            continue

        # Model inference
        sr_np = run_inference(model_obj, sample)

        # Denormalize HR
        hr_np = ( sample["hr"][0].numpy() * HR_STD) + HR_MEAN

        # Valid pixels
        region_mask = region_mask.numpy()
        valid = ( region_mask & np.isfinite(hr_np) & np.isfinite(sr_np))

        if valid.sum() == 0:
            continue

        # Residual
        residual = sr_np[valid] - hr_np[valid]

        all_residuals.append(residual)
        all_sample_ids.extend([idx] * len(residual))

    # Flatten
    residual_v = np.concatenate(all_residuals)
    sample_ids_v = np.array(all_sample_ids)

    # Save
    if save_path is None: save_path = ( f"{region}_residuals_" f"{model_name.lower().replace(' ', '_')}.npz")

    np.savez_compressed(
        save_path,
        sample_id=sample_ids_v,
        residual=residual_v,
        model_name=model_name,
        region=region
    )

    print(f"Saved {len(residual_v):,} " f"{region} residuals for {model_name}")

    return pd.DataFrame({
        "sample_id": sample_ids_v,
        "residual": residual_v,
        "model": model_name,
        "region": region
    })